# Setup
## Dependencies

In [44]:
from pathlib import Path
import re
import sqlite3

## File Locations

In [45]:
mosaic_path = Path("../output mosaics/non-flats")
flat_mosaic_path = Path("../output mosaics/flats")

filename_list = sorted(mosaic_path.glob("*-mosaic.fits"))
flat_filename_list = sorted(flat_mosaic_path.glob("*-mosaic.fits"))

master_db_path = Path("../databases/Master_Database.db").resolve()
master_conn = sqlite3.connect(master_db_path)

In [55]:
batch_log_folder = Path("../lab batches/flat_01/logs")
backup_folder = Path("../backups")

write_errors = True

## Functions

In [47]:
def get_group_id(filename_list):
    mosaic_records = []

    for files in filename_list:
        filename = files.name
        match = re.search(r"-g(\d+)-mosaic\.fits$", filename)

        if match is None:
            print("no ID, skipping:", filename)
            continue

        group_id = int(match.group(1))
        mosaic_records.append((group_id, filename))

    return(mosaic_records)

In [48]:
def insert_mosaic_names(master_conn, mosaic_records):
    
    sql = """
        UPDATE observations
        SET mosaic_name = ?
        WHERE group_id = ?
          AND mosaic_name IS NULL
    """
    updated = 0
    with master_conn:
        for group_id, filename in mosaic_records:
            cursor = master_conn.execute(sql, (filename, group_id))
            updated += cursor.rowcount
    return updated

In [49]:
def insert_flat_mosaic_names(master_conn, mosaic_records):
    sql = """
        UPDATE observations
        SET mosaic_name_f = ?
        WHERE group_id = ?
          AND mosaic_name_f IS NULL
    """
    updated = 0
    with master_conn:
        for group_id, filename in mosaic_records:
            cursor = master_conn.execute(sql, (filename, group_id))
            updated += cursor.rowcount
    return updated

In [50]:
def get_mosaic_error(log_path, flat_used):
    text = log_path.read_text(errors="replace")

    if flat_used and "Flatfield found:" not in text:
        return "mosaic: flat not applied"

    if "minvalue must be less than or equal to maxvalue" in text:
        return "mosaic: plot crash"

    if "Less than 5 stars" in text:
        return "mosaic: too few stars"

    if "starfit" in text:
        return "mosaic: starfit failed"

    return "mosaic: unknown failure"

In [51]:
def get_failed_groups(master_conn, log_folder, flat_used):
   if flat_used:
       sql = """
           SELECT group_id FROM observations
           WHERE mosaic_name_f IS NULL AND error_f IS NULL
       """
   else:
       sql = """
           SELECT group_id FROM observations
           WHERE mosaic_name IS NULL AND error IS NULL
       """

   pending_ids = []
   for row in master_conn.execute(sql).fetchall():
       pending_ids.append(row[0])

   failed = []
   for group_id in pending_ids:
       log_path = log_folder / f"g{group_id}.log"

       # No log means the group was skipped (no flat), so it stays pending
       if not log_path.is_file():
           continue

       label = get_mosaic_error(log_path, flat_used)
       failed.append((group_id, label))

   return failed

In [52]:
failed_groups = get_failed_groups(master_conn, batch_log_folder, flat_used=True)

label_counts = {}
for group_id, label in failed_groups:
   label_counts[label] = label_counts.get(label, 0) + 1

print("Failed groups:", len(failed_groups))
for label, count in sorted(label_counts.items()):
   print("  ", label, count)

for group_id, label in failed_groups:
   print(group_id, label)

Failed groups: 0


# Actual Code Execution

In [53]:
mosaic_records = get_group_id(filename_list)
num_done = insert_mosaic_names(master_conn, mosaic_records)

flat_mosaic_records = get_group_id(flat_filename_list)
flat_num_done = insert_flat_mosaic_names(master_conn, flat_mosaic_records)
print(f"Non flat: {num_done}")
print(f"With flat: {flat_num_done}")


Non flat: 0
With flat: 151


In [56]:
from datetime import datetime

def insert_flat_mosaic_errors(master_conn, failed_groups):
   sql = """
       UPDATE observations
       SET error_f = ?
       WHERE group_id = ?
         AND mosaic_name_f IS NULL
         AND error_f IS NULL
   """
   updated = 0
   with master_conn:
       for group_id, label in failed_groups:
           cursor = master_conn.execute(sql, (label, group_id))
           updated += cursor.rowcount
   return updated


if write_errors:
   backup_folder.mkdir(exist_ok=True)
   stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
   backup_path = backup_folder / f"Master_Database.before_flat_errors_{stamp}.db"

   backup_conn = sqlite3.connect(backup_path)
   master_conn.backup(backup_conn)
   backup_conn.close()
   print("Backup saved to:", backup_path.resolve())

   num_errors = insert_flat_mosaic_errors(master_conn, failed_groups)
   print("Errors written:", num_errors)
else:
   print("Preview only. Set write_errors = True to write.")

Backup saved to: /Users/mehtabsingh/Desktop/Desktop/Durham/3rd Year/Modules/Astro Lab/Astrolab_Code/backups/Master_Database.before_flat_errors_20261026_164613.db
Errors written: 0
